# Comparing Safety-Removal Subspaces in Aligned LLMs

End-to-end Colab notebook for reproducing all experiments.

---

## Research question

Modern aligned LLMs refuse harmful requests because RLHF/SFT training instills a **refusal subspace** — a low-dimensional region of the model's activation space that triggers refusal behavior. Several recent methods try to remove this behavior by identifying and excising that subspace. We ask two interconnected questions:

1. **Do the methods converge?** When DIM, ActSVD, and RDO each find a "safety subspace," are they identifying the same geometric structure, or do they find distinct solutions?
2. **Is safety entangled with utility?** Does the refusal subspace overlap with the activation subspace the model uses for normal (harmless) instruction following? If it does, a safety-removal intervention that preserves utility is geometrically impossible — the two signals cannot be cleanly separated.

The answers matter for alignment: if all methods target the same direction, a single targeted defense could block all of them. If safety and utility are geometrically entangled, safety removal inevitably degrades model capability.

---

## Methods compared

| Method | Core idea | Intervention type |
|--------|-----------|-------------------|
| **DIM** (Arditi et al. 2024) | Compute the mean activation difference between harmful and harmless prompts at each layer; pick the single layer/direction that best separates them via linear probe accuracy | Activation-space ablation: at inference time, zero out the component of every residual stream vector along the direction |
| **ActSVD** (Wei et al. 2024) | Decompose each weight matrix via SVD; identify singular vectors that respond most to the harmful/harmless contrast; project them out of the weight matrix permanently | Weight-space surgery: saves a modified copy of the model, no hooks at inference |
| **RDO/RCO** (Wollschläger et al. 2025) | Starting from the DIM direction, gradient-descent optimizes a direction (or 2D cone) to maximize a jailbreaking objective while staying close to the initial DIM estimate | Activation-space ablation: same hook mechanism as DIM, but direction chosen by optimization rather than statistics |

---

## Analysis pipeline

```
Train methods                 Evaluate behavior                Analyze geometry
─────────────────             ─────────────────────────        ────────────────────────────────────
DIM   (cell 3)   ──────────►  Behavioral benchmark  (cell 6)
ActSVD (cell 4)  ──────────►    JailbreakBench ASR
RDO/RCO (cell 5) ──────────►    Harmless compliance            Safety-vs-utility MSO (cell 7)
                                Perplexity (utility proxy)     Method-vs-method MSO  (cell 8)
                                                               RepInd profile        (cell 9)
                                               Prompt attack probe   (cell 10)
```

The **behavioral benchmark** (cell 6) answers: *how effective is each method at enabling jailbreaks, and at what cost to utility?*

The **geometric analyses** (cells 7–9) answer: *why* each method works or doesn't, by measuring the shape and location of the safety subspace each method identifies.

---

## Key metric: Maximum Subspace Overlap (MSO)

Given two subspaces A (rank k_A) and B (rank k_B) in the same ambient space, MSO measures how much of A's energy lies inside B:

$$\text{MSO}(A, B) = \frac{1}{k_A} \| U_A^T U_B \|_F^2$$

where U_A, U_B are orthonormal bases. The random baseline is k_B / d (d = ambient dimension). Values well above the baseline mean the two subspaces share structure beyond chance.

We use MSO to ask:
- Does the DIM/RCO direction lie in the utility activation subspace? (cell 7)
- Does the DIM direction lie in the ActSVD weight-change subspace? (cell 8)
- Does the RCO direction lie in the ActSVD weight-change subspace? (cell 8)

---

## GPU requirements
| Step | Minimum | Recommended |
|------|---------|-------------|
| DIM | 16 GB (T4) | A100 40 GB |
| ActSVD | 24 GB | A100 80 GB |
| RDO training | 24 GB | A100 40 GB |
| Benchmark eval (base/DIM/ActSVD) | 16 GB | A100 40 GB |
| Safety-utility overlap | 16 GB | A100 40 GB |
| Method overlap (MSO) | CPU only | CPU only |
| RepInd analysis | 16 GB | A100 40 GB |

**Recommended**: Use an A100 runtime. Go to `Runtime → Change runtime type → GPU → A100`.

## Quick smoke test
Set `MODEL_PATH = "google/gemma-2b-it"` and skip ActSVD (cell 4). DIM + RDO + all analysis runs in ~45 min on a T4.

In [ ]:
from pathlib import Path
import os, subprocess

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

REPO_URL = "https://github.com/evan203/nlp-project-proposal.git"
BRANCH = "experiment/kyle"
PROJECT = Path('/content/nlp-project-proposal')

# --- Model selection ---
# For the full report: meta-llama/Llama-3.1-8B-Instruct (requires A100 + HF token)
# For a quick smoke test: google/gemma-2b-it (no token needed, T4-compatible)
MODEL_PATH = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_ALIAS = MODEL_PATH.split('/')[-1]  # e.g. Llama-3.1-8B-Instruct

HF_HOME = PROJECT / 'code' / 'llm_weights'
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME)
os.environ['PYTHON_RUNNER'] = 'python'

def run(cmd, cwd=None, env_extra=None):
    """Run a shell command and stream output line-by-line."""
    env = os.environ.copy()
    if env_extra:
        env.update({k: str(v) for k, v in env_extra.items()})
    effective_cwd = cwd if cwd is not None else (PROJECT if PROJECT.exists() else None)
    print(f"\n$ {cmd}")
    with subprocess.Popen(
        cmd, shell=True, cwd=effective_cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end='')
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)

print('GPU status:')
run('nvidia-smi')

## 1. Clone the repository and authenticate Hugging Face

Llama-3.1-8B-Instruct is gated. Paste a token that has accepted the Meta Llama license at [huggingface.co/meta-llama](https://huggingface.co/meta-llama).

We use **Llama-3.1-8B-Instruct** as the primary model because:
- It is the canonical base for Wollschläger et al. (2025) and Arditi et al. (2024)
- The 8B size fits in A100 40 GB memory with reasonable throughput
- The instruct-tuned version has strong RLHF-induced refusal behavior, making the safety subspace easier to study

If using `google/gemma-2b-it`, skip the HF login step.

In [ ]:
if not PROJECT.exists():
    run(f"git clone -b {BRANCH} {REPO_URL} {PROJECT}", cwd="/content")
else:
    run(f"git checkout {BRANCH} && git pull", cwd=PROJECT)

run("pip -q install -U huggingface_hub")

from huggingface_hub import notebook_login, whoami

try:
    user = whoami()
    print("Already logged into Hugging Face as:", user["name"])
except Exception:
    notebook_login()

## 2. Install dependencies

- `transformers` / `accelerate` / `torch`: model loading and inference
- `datasets`: loads JailbreakBench, harmless splits, wikitext-2, and alpaca for evaluation
- `nnsight==0.3.7`: **pinned** because RDO/cones use an internal tuple-indexing API (`layer.output[i]`) that changed in 0.4.x; newer versions return 2-tuples from self-attention instead of 3-tuples
- `safetensors`: lets the method-overlap script load individual weight tensors from sharded checkpoints without instantiating the full model in GPU RAM
- `vllm`: imported by some JailbreakBench evaluation utilities (not used directly, but required for import-time side-effects)

In [ ]:
run('pip -q install -U torch torchvision transformers accelerate datasets '
    'sentencepiece protobuf tqdm jaxtyping matplotlib seaborn '
    'zstandard litellm einops safetensors')
run('pip -q install -U nnsight==0.3.7')
run('pip -q install -U vllm')  # required by evaluate_jailbreak.py imports
run('pip -q install python-dotenv wandb')  # required by RDO training

# Verify GPU is accessible
run("python -c 'import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))'")  # noqa

## 3. Run DIM (Difference-in-Means)

**What it does**: DIM collects residual-stream activations at end-of-instruction (EOI) token positions for harmful and harmless prompts, then computes the mean difference per layer. Each layer's mean-diff vector is a candidate refusal direction. A linear probe is trained on each candidate; the layer with the best probe accuracy is selected as the final direction. At inference, the ablation hook projects every residual stream vector onto the plane orthogonal to this direction (i.e., zeros out the refusal component).

**Why we run it first**: DIM is both the simplest method and the initialization point for RDO. Its `mean_diffs.pt` file (all per-layer candidates) feeds into the safety-utility overlap analysis, and its `direction.pt` (the selected single direction) is used to initialize RDO's optimizer.

**What to look at in the outputs**:
- `direction_metadata.json` tells you which layer was selected — typically layer 13–15 for Llama-3.1-8B
- `mean_diffs.pt` has shape `[n_prompts, n_layers, d_model]`; averaged over prompts it becomes the safety subspace estimate in cell 7

Saves:
- `code/methods/dim/pipeline/runs/<model>/direction.pt` — selected refusal direction
- `code/methods/dim/pipeline/runs/<model>/generate_directions/mean_diffs.pt` — per-layer mean-diff candidates
- `code/methods/dim/pipeline/runs/<model>/direction_metadata.json` — selected layer + position
- `code/methods/dim/pipeline/runs/<model>/completions/` — baseline/ablation completions

**Time estimate**: ~20 min on A100 for Llama-3.1-8B.

In [ ]:
run('./scripts/run_dim.sh', env_extra={'MODEL_PATH': MODEL_PATH})

## 4. Run ActSVD

**What it does**: ActSVD decomposes each weight matrix W via SVD: W = U Σ V^T. It then measures the correlation between each left singular vector u_i and the harmful/harmless activation contrast vector. Singular vectors with high alignment to the safety signal are projected out of W, producing a modified weight W' = W - Σ_i α_i u_i u_i^T v_i. This is applied across all attention and MLP projection matrices. The result is a permanently modified model stored on disk.

**Key hyperparameters**:
- `RANK_POS` / `RANK_NEG`: how many singular vectors to project out from the safety (harmful-correlating) and utility (harmless-correlating) directions, respectively. Higher values remove more; too high degrades utility.
- `NSAMPLES`: how many prompts to use for estimating the activation contrast. 128 is the standard; 32 is a smoke-test shortcut.

**Why it matters for our analysis**: ActSVD is the only method that permanently edits weights rather than applying inference-time hooks. This means we compare it to DIM/RDO not by comparing direction vectors directly, but by asking: *does the DIM/RDO direction lie within the subspace ActSVD modified?* A high MSO would mean all three methods are effectively targeting the same structure, just via different intervention mechanisms.

Saves a modified model to `code/methods/actsvd/out/`.

**Time estimate**: ~40 min on A100 with `NSAMPLES=128`.

**Smoke test**: set `NSAMPLES=32` to reduce time and VRAM to ~16 GB.

In [ ]:
# ActSVD with paper-optimal hyperparams (Wei et al. 2024 Table 6 best config).
# Their optimal: r^u=3950 (utility), r^s=4090 (safety), effective Delta-W rank ~6.
# Our previous setting (r^u=3000, r^s=4000) was much more aggressive and hurt PPL.
run('./scripts/run_actsvd.sh', env_extra={
    'MODEL_ALIAS': MODEL_ALIAS,
    'NSAMPLES':    '128',
    'RANK_POS':    '3950',  # utility rank r^u  (paper optimal)
    'RANK_NEG':    '4090',  # safety rank  r^s  (paper optimal)
    'EVAL_PPL':    '0',
    'EVAL_ATTACK': '0',
})

## 5. Run RCO (Refusal Cone Optimization)

**What it does**: Starting from the DIM direction (which is why DIM must run first), RCO uses gradient descent to optimize a 2-dimensional *refusal cone* — a subspace spanned by two orthogonal directions. The cone is the paper's main contribution: at inference time, the model's activations are ablated along whichever direction within the cone best suppresses refusal for a given prompt. This makes the intervention robust to adversarial prompt variation: a prompt engineered to escape one direction in the cone may still be caught by another.

**Why a cone, not a single direction**: A single optimized direction (RDO) is a 1D ablation — an adversary can craft prompts that activate refusal behavior orthogonally to that direction, evading the intervention entirely. A 2D cone removes an entire plane from the activation space, closing that escape route. The cone's second basis vector is initialized orthogonally to the first and trained to cover the residual variance in refusal behavior that the first vector misses.

**Why RCO builds on DIM**: The gradient optimization is initialized at the DIM direction. Without a good starting point, the loss landscape is highly non-convex. DIM provides a statistically motivated initialization already close to the true refusal axis; RCO's optimizer then refines it via gradient signal from actual jailbreak attempts.

**Geometric significance for our analysis**: The cone gives us a 2D safety subspace. When we compute MSO against the utility subspace (cell 7) and ActSVD's weight-change subspace (cell 8), we use the first basis vector. The second vector's coverage is visible in the heatmap at rank k=2. If the second vector adds substantial utility overlap beyond the first, it suggests the safety signal is genuinely multi-dimensional.

**Steps inside `run_rco_eval.sh`**:
1. **Train** `rdo.py` in cone mode (gradient optimization of 2 orthogonal directions, saves checkpoints)
2. **Extract** the best checkpoint → `code/results/geometry_repind/rco_direction.pt`
3. **Evaluate** on JailbreakBench + harmless compliance → appends `METHOD_NAME` to `benchmark_results.json`
4. **RepInd** comparison (DIM + RCO directions) → saved to `code/results/geometry_repind_rco/`

**Time estimate**: ~90 min on A100 (cone mode). See Appendix A to run the faster single-direction variant instead.

In [ ]:
run('./scripts/run_rco_eval.sh', env_extra={
    'MODEL': MODEL_PATH,
    'MODE': 'cone',
    'CONE_DIM': '2',
    'METHOD_NAME': 'RCO-Cone-2',
    'WANDB_MODE': 'offline',
    'SKIP_EVAL': '0',      # eval_direction_benchmark.py runs inside this script
    'SKIP_REPIND': '0',
    'SAVE_MODIFIED': '0',
})

## 6. Evaluate base model, DIM, ActSVD, and a random-direction baseline

**What we measure and why**:

| Metric | How computed | What it tells us |
|--------|-------------|------------------|
| **JailbreakBench ASR (substring)** | Fraction of 100 JBB harmful prompts where the model does *not* produce a refusal substring | Cheap default judge. Sensitive to refusal prefix language. |
| **JailbreakBench ASR (LLM judge)** | All saved completions re-graded in a *separate* post-hoc pass by the **unmodified base model** (cell 13b below). | Catches Llama-3's "I'd be happy to help, actually I can't…" pattern. Using the base model as judge for *every* method's completions avoids the cross-method confound where a modified model's own intervention biases its self-judgment. |
| **Harmless compliance** | Fraction of 100 harmless prompts where the model answers (does not refuse) after ablation | Detects over-ablation. Should stay near 1.0. |
| **Perplexity (pile/alpaca)** | Next-token NLL on wikitext-2 + alpaca with hooks applied | Utility proxy. Higher = worse. |
| **TruthfulQA truthful rate** | 64 questions, greedy generation, substring match against `correct_answers` / `incorrect_answers` | Side-effect test where Wollschläger et al. claim DIM does worse than RCO. |
| **Bootstrap 95% CI** | 1,000 percentile-bootstrap samples on ASR and harmless compliance | Quantifies uncertainty so we can talk about whether 0.62 vs 1.00 is a real gap. |

**Random-direction baseline** is included as a sanity check: ablating a random unit vector should leave ASR near baseline (~0.15). If it doesn't, our entire ablation story is suspect.

ActSVD now uses the **paper-optimal hyperparameters** from Wei et al. (`r^u=3950`, `r^s=4090`, effective $\Delta W$ rank ~6) instead of our earlier aggressive settings — this should bring perplexity down toward Pile baseline while keeping ASR comparable.

RCO-Cone-2 was already evaluated inside `run_rco_eval.sh` (step 5). All results append to `code/results/benchmark/benchmark_results.json`. Re-running this cell overwrites entries.

The **post-hoc LLM judge runs in cell 13b** below — once *all* methods' completions are saved, we load the unmodified base model exactly once and grade every saved JSON in a single pass.

In [ ]:
import os
CODE = PROJECT / 'code'
DIM_DIR_RUN = CODE / f'methods/dim/pipeline/runs/{MODEL_ALIAS}'

# Common args appended to every eval invocation:
#   --eval_ppl --n_ppl_samples 64           (perplexity)
#   --eval_truthfulqa --n_tqa_samples 64    (TruthfulQA side-effect test)
#   --bootstrap 1000                         (95% CI on ASR + harmless compliance)
# NOTE: NO --llm_judge here.  The LLM judge is run as a separate post-hoc
# pass in cell 13b using the unmodified base model — this avoids the
# cross-method confound where a modified model's own intervention biases
# its self-judgment.
COMMON_ARGS = (
    ' --eval_ppl --n_ppl_samples 64'
    ' --eval_truthfulqa --n_tqa_samples 64'
    ' --bootstrap 1000'
)

# Base model — no intervention
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --method_name "Base ({MODEL_ALIAS})"'
    f' --no_ablation' + COMMON_ARGS,
    cwd=CODE,
)

# DIM — direction ablation
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --direction_path {DIM_DIR_RUN}/direction.pt'
    f' --direction_metadata {DIM_DIR_RUN}/direction_metadata.json'
    f' --method_name DIM-Ablated' + COMMON_ARGS,
    cwd=CODE,
)

# ActSVD — load modified model directly (no direction hooks)
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --modified_model_path methods/actsvd/out'
    f' --method_name ActSVD-Modified' + COMMON_ARGS,
    cwd=CODE,
)

# Random-direction baseline — sanity check that ablating an arbitrary unit
# vector does NOT raise ASR. This is the standard counterfactual missing
# from the original paper set.
run(
    f'python analysis/eval_direction_benchmark.py'
    f' --model_path {MODEL_PATH}'
    f' --random_direction --seed 7'
    f' --method_name Random-Direction-7' + COMMON_ARGS,
    cwd=CODE,
)

# Show current state of benchmark_results.json
import json, pathlib
benchmark_path = CODE / 'results/benchmark/benchmark_results.json'
if benchmark_path.exists():
    benchmark = json.loads(benchmark_path.read_text())
    print('\nbenchmark_results.json entries (substring judge — LLM judge added by cell 13b):')
    for name, entry in benchmark.items():
        jbb = entry.get('jailbreakbench', {})
        asr = jbb.get('asr', 'N/A')
        hc  = entry.get('harmless_compliance', {}).get('rate', 'N/A')
        tqa = entry.get('truthfulqa', {}).get('truthful_rate')
        ppl_pile = entry.get('perplexity', {}).get('pile', {}).get('perplexity', 'N/A')
        suffix = f', tqa={tqa}' if tqa is not None else ''
        print(f'  {name}: ASR={asr}, harmless={hc}, ppl_pile={ppl_pile}{suffix}')

### 6b. Post-hoc LLM judge (consistent across methods)

Loads the **unmodified base Llama** exactly once and grades every saved `*_jbb_ablation_completions.json` with the same judge, in one pass.

**Why this is separate**: using the *loaded* model as judge inside `eval_direction_benchmark.py` is unsafe because for DIM-ablated, ActSVD-modified, and RCO-cone runs, the loaded model has hooks active or modified weights — the very intervention that compromised refusal *also* compromises the judge's ability to detect refusal. By running the judge as a separate post-hoc pass on a clean base model, every method gets graded by the same unmodified judge.

**Same-family-as-judge limitation**: Llama judging Llama still has self-bias. The cross-method confound is removed; the same-family confound remains. We document this in the report's Limitations section.

This adds ~5–8 minutes to total runtime: one extra forward pass over ~400 completions (4 methods × 100 JBB rows each), max_new_tokens=6.

In [ ]:
run(
    f'python analysis/judge_completions.py'
    f' --model_path {MODEL_PATH}'
    f' --benchmark_dir results/benchmark'
    f' --batch_size 8'
    f' --bootstrap 1000',
    cwd=CODE,
)

# Show updated benchmark_results.json with both judges
import json
benchmark_path = CODE / 'results/benchmark/benchmark_results.json'
if benchmark_path.exists():
    benchmark = json.loads(benchmark_path.read_text())
    print('\nbenchmark_results.json after post-hoc judge:')
    print(f'{"Method":<32} {"sub-ASR":>8} {"LLM-ASR":>8} {"harmless":>9} {"ppl_pile":>9}')
    for name, entry in benchmark.items():
        jbb = entry.get('jailbreakbench', {})
        sub = jbb.get('asr')
        llm = jbb.get('asr_llm_judge')
        hc  = entry.get('harmless_compliance', {}).get('rate')
        ppl = entry.get('perplexity', {}).get('pile', {}).get('perplexity')
        sub_s = f'{sub:.2f}' if isinstance(sub, (int,float)) else str(sub)
        llm_s = f'{llm:.2f}' if isinstance(llm, (int,float)) else 'N/A'
        hc_s  = f'{hc:.2f}'  if isinstance(hc,  (int,float)) else str(hc)
        ppl_s = f'{ppl:.2f}' if isinstance(ppl, (int,float)) else str(ppl)
        print(f'{name:<32} {sub_s:>8} {llm_s:>8} {hc_s:>9} {ppl_s:>9}')

## 7. Safety-vs-utility activation overlap

**Core question**: Is the refusal direction geometrically entangled with the model's utility representations?

**How it works**: We collect residual-stream activations from harmless prompts at end-of-instruction positions and compute the top-k PCA directions — this is the "utility subspace." We then measure what fraction of each method's refusal direction's energy lies within that subspace (MSO, see intro for formula). A high MSO means the refusal direction is a direction the model also uses for normal instruction following; removing it would therefore also impair utility.

**Why this matters for the research question**: If DIM, RDO, and ActSVD's directions all have *low* MSO with utility, they all successfully target a safety-specific subspace orthogonal to utility — a geometrically clean separation. If they all have *high* MSO, safety and utility are entangled, and any effective safety removal will be lossy for utility. The perplexity results from cell 6 are the behavioral validation of this geometric prediction.

**Directions overlaid in the per-layer chart**:
- **DIM mean-diffs** (bar chart) — averaged per-layer safety vector; the main estimate of the safety subspace
- **DIM selected direction** — the single layer's direction DIM picked; typically more concentrated in the selected layer
- **RCO direction** — the optimized direction from step 5 (overlaid as a line if `rco_direction.pt` exists)
- **ActSVD activation delta** — per-layer mean shift in activations caused by ActSVD weight editing (extracted automatically if `results/actsvd/out` exists)

Showing all four on the same axes lets us see if different methods identify the same high-overlap layers.

Outputs in `code/results/safety_utility_overlap/`:
- `safety_utility_overlap_results.json`
- `safety_utility_overlap_per_layer.png` — primary figure: bar chart + overlaid method lines
- `safety_utility_overlap_heatmap.png` — rank × layer MSO heatmap (shows how overlap changes as you include more utility PCA components)
- `safety_utility_overlap_by_rank.png` — mean MSO vs utility PCA rank k

**Time estimate**: ~15 min on A100.

In [ ]:
run('./scripts/run_safety_utility_overlap.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'N_UTILITY_SAMPLES': '128',
    'UTILITY_RANKS': '1,2,4,8,16,32',
    'PRIMARY_RANK': '8',
    'ACTSVD_OUT': str(CODE / 'methods/actsvd/out'),
})

## 8. Method subspace overlap (MSO)

**Core question**: Do DIM and RDO identify directions that lie within the subspace ActSVD modifies? Are DIM and RDO geometrically aligned with each other?

**Why this is the central comparison**: DIM and RDO work by ablating a *direction in activation space*. ActSVD works by modifying *weight matrices*. These are different levels of the computation graph. If DIM/RDO's direction lies in the subspace spanned by ActSVD's weight changes (measured as the SVD of ΔW = W_actsvd − W_base at each layer and weight type), it would mean all three methods are targeting the same fundamental safety structure, just expressed differently: as an activation direction vs. as a change to the weight matrices that implement that direction.

**What high MSO (DIM/RDO vs ActSVD) means**: The safety axis is a consistent feature of the model — it has a geometric signature that appears both in the weight matrices and in the activation space. This would suggest a strong, localized safety mechanism that multiple approaches independently discover.

**What high cosine similarity (DIM vs RDO) means**: RDO's gradient optimization converged to approximately the same direction DIM found statistically. This would validate DIM's simple mean-difference approach: the statistically optimal direction is also the loss-optimal direction.

This is a CPU-only computation: it loads weight matrices directly from safetensors files without instantiating the full model in GPU memory.

Outputs in `code/results/method_overlap/`:
- `DIM_vs_ActSVD_per_layer_mso_heatmap.png` / `_mso_per_layer.png` — per-layer × weight-type MSO grid
- `RCO_vs_ActSVD_per_layer_mso_heatmap.png` / `_mso_per_layer.png` — same for RCO
- `direction_cosine.png` — cosine similarities between direction pairs (DIM vs RCO, etc.)
- `cross_model_cosine.png` — cross-model DIM direction similarities (pre-computed, shows universality)

**Time estimate**: ~5 min (CPU, no model loading).

In [ ]:
run('./scripts/run_method_overlap.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'ACTSVD_MODEL_PATH': str(CODE / 'methods/actsvd/out'),
})

## 9. RepInd profile analysis

**Core question**: Is the refusal subspace *causally* independent of utility representations, or do they share the same causal pathway through the model?

**What RepInd measures**: Representational independence (RepInd) is a causal intervention test. It patches activations along the refusal direction between a harmful and harmless prompt pair, then measures the downstream effect on both harmful and utility behavior. A direction is representationally independent if patching it changes refusal behavior without changing utility behavior — i.e., the two behaviors have separable causal mediators.

**Why this goes beyond MSO**: MSO measures geometric overlap (correlation). A direction could have low MSO with utility yet still share causal pathways — for instance, if the refusal direction modulates a shared attention head that also handles instruction following. RepInd catches these functional dependencies that pure geometry misses.

**Connection to the research question**: If DIM and RCO have similar RepInd profiles, they are causally equivalent interventions — they tap the same causal mechanism, just from different directions. If they differ, one method may be more surgically precise even if their directions appear geometrically similar.

The RCO RepInd comparison (DIM + RCO directions) was already saved to `code/results/geometry_repind_rco/` by step 5. This cell re-runs the DIM-derived standalone profile as a reference.

To run RepInd using the trained RCO directions instead:
```python
run('./scripts/run_geometry_repind.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'DIRECTIONS_JSON': 'code/results/geometry_repind/directions_rco.json',
    'OUTPUT_DIR': 'code/results/geometry_repind_rco',
})
```

**Time estimate**: ~10 min on A100.

In [ ]:
run('./scripts/run_geometry_repind.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'N_PROMPTS': '32',
    'BATCH_SIZE': '8',
    'DERIVED_CONE_DIM': '3',
})

## 10. Probe: does adversarial wrapping suppress the refusal direction?

**Core question**: When adversarial prompts successfully jailbreak the base model *without any weight or activation modification*, do they work by suppressing the same refusal direction that DIM/ActSVD/RCO explicitly ablate — or through an orthogonal mechanism?

**Why it matters for our analysis**:
Cells 7-9 established the geometric picture: DIM's selected direction has a utility-MSO of **0.078** (40× random), while RCO's optimized direction cuts that to **0.004** (1.8× random) — nearly orthogonal to the utility subspace. The two directions are correlated (cosine ≈ 0.450) but not identical. The question now is whether *prompt-level* attacks that bypass the base model's safety (base ASR = 0.15) target the same geometric channel.

Three core questions plus two new diagnostic experiments:

| # | Question | Evidence |
|---|----------|----------|
| **Q1** | Do adversarial wrappings suppress the DIM direction (utility-MSO 0.078, layer 11) compared to bare harmful requests? | `proj_DIM`: lower for adversarial_harmful vs direct_request? |
| **Q2** | Do they suppress RCO's more surgical direction (utility-MSO 0.004) differently from DIM? | `proj_RCO`: if DIM and RCO projections diverge, prompt attacks exploit different geometry than RCO |
| **Q3** | Does adversarial wrapping on *benign* content (control) show a different projection profile? | `adversarial_benign` group: the same wrapper style on non-harmful instructions should leave the refusal direction unaffected |
| **(NEW) Layer sweep** | Where in the network does the prompt attack suppress the direction? | One forward pass with hooks at every layer → projection trajectory per group |
| **(NEW) Ablation cross-test** | For the same prompts, does DIM ablation succeed where prompt attacks fail? | Generate completions twice (base + DIM-ablated) and compare per-group ASR |

**Data sources** (real attack text, not synthetic templates):

| Group | Source | Description |
|-------|--------|-------------|
| `direct_request` | **HarmBench** standard behaviors, local JSON | Bare harmful instructions with no wrapping — baseline, model should refuse almost all |
| `adversarial_harmful` | **WildJailbreak** `train` config, `adversarial_harmful` rows | Jailbreak-wrapped versions of harmful requests, collected from the wild |
| `adversarial_benign` | **WildJailbreak** `train` config, `adversarial_benign` rows | Same adversarial wrapper style applied to harmless requests — control group |

WildJailbreak is streamed (no full 262k-row download) with a shuffle buffer so samples are drawn uniformly. HarmBench loads from the local JSON already bundled with the DIM method — no network call needed.

**What we measure**:
- **Direction projection** (`proj_DIM`, `proj_RCO`): single forward pass captures the residual stream at the DIM-selected layer (layer 11) at the end-of-instruction (EOI) token position. That activation is projected onto the unit direction vector. Lower projection = the prompt pushes the model's internal state away from the refusal subspace.
- **Layer sweep** (NEW): same projection at *every* block input. Tells us whether suppression is layer-localized.
- **ASR**: whether the model's generation contains a refusal substring (same judge as cell 6).
- **Ablation ASR** (NEW): same prompts, but generated through the DIM-ablated model. Lets us compare prompt-attack mechanism vs direction-removal mechanism on identical inputs.
- **Bootstrap 95% CI** on per-group ASR and projection means.

**Outputs**: seven figures in `results/probe_attack_types/` and `docs/figures/`:
1. Scatter — projection vs. jailbreak outcome per prompt (one panel per direction)
2. Box plots — projection distributions split by refused/jailbroken per group
3. Bar charts — ASR and mean projection per group (side-by-side)
4. DIM vs RCO projection scatter — one point per prompt; reveals whether attacks that suppress DIM also suppress RCO
5. Correlation scatter — mean projection vs. ASR per group
6. **Layer sweep (NEW)** — per-layer mean projection per group with ±SE bands
7. **Ablation cross-test (NEW)** — base ASR vs DIM-ablated ASR side-by-side per group

**Requires**: DIM (cell 3) for `direction.pt` + `direction_metadata.json`. RCO (cell 5) for `proj_RCO`. `HF_TOKEN` Colab secret with WildJailbreak access.

**Time estimate**: ~20 min on A100 with all extras enabled. Set `LAYER_SWEEP=0` or `ABLATION_CROSS=0` in the env_extra dict below to skip individual extras.

In [ ]:
run('./scripts/run_probe_attack_types.sh', env_extra={
    'MODEL_PATH':      MODEL_PATH,
    'MODEL_ALIAS':     MODEL_ALIAS,
    'N_DIRECT':        '25',   # HarmBench direct-request baseline prompts
    'N_PER_TYPE':      '25',   # WildJailbreak prompts per group
    'BATCH_SIZE':      '8',
    'MAX_NEW_TOKENS':  '200',
    'LAYER_SWEEP':     '1',     # 1 = per-layer projection sweep (NEW)
    'ABLATION_CROSS':  '1',     # 1 = also generate completions with DIM ablated (NEW)
    'BOOTSTRAP':       '1000',  # 0 = disable bootstrap CIs (NEW)
})

# Print per-group summary
import json as _json
from collections import defaultdict as _dd
probe_path = CODE / 'results/probe_attack_types/results.json'
if probe_path.exists():
    _results = _json.loads(probe_path.read_text())
    for _r in _results:
        if 'tactic' in _r and 'attack_type' not in _r:
            _r['attack_type'] = _r['tactic']

    _proj_keys = sorted({k for _r in _results for k in _r if k.startswith('proj_')})
    _has_abl = any('ablated_is_jailbreak' in _r for _r in _results)
    _by_type = _dd(list)
    for _r in _results:
        _by_type[_r['attack_type']].append(_r)

    _hdr = f'{"Group":<26} {"n":>3} {"ASR":>5}'
    if _has_abl:
        _hdr += f'  {"AblASR":>6}'
    for _pk in _proj_keys:
        _dn = _pk.replace('proj_', '')
        _hdr += f'  {f"proj_{_dn}":>14}'
    print(_hdr)
    print('-' * len(_hdr))

    _order = ['direct_request', 'adversarial_harmful', 'adversarial_benign']
    for _atype in _order:
        if _atype not in _by_type:
            continue
        _entries = _by_type[_atype]
        _n = len(_entries)
        _asr = sum(e['is_jailbreak'] for e in _entries) / _n
        _row = f'{_atype:<26} {_n:>3} {_asr:>5.2f}'
        if _has_abl:
            _abl = sum(e.get('ablated_is_jailbreak', 0) for e in _entries) / _n
            _row += f'  {_abl:>6.2f}'
        for _pk in _proj_keys:
            _mp = sum(e.get(_pk, 0) for e in _entries) / _n
            _row += f'  {_mp:>14.4f}'
        print(_row)

## 11. Regenerate all figures and sync to docs

Runs all plotting scripts from the latest JSON results and syncs output PNGs to `docs/figures/` for the Typst report. At this point all three analysis streams (behavioral benchmark, safety-utility overlap, method overlap) have populated their respective JSON files; this cell just re-renders the visuals.

Safe to re-run at any point — all plots are idempotent given the same JSON inputs.

In [ ]:
import pathlib

CODE_DIR = PROJECT / 'code'

# Benchmark: all methods now in benchmark_results.json
run('python analysis/plot_benchmarks.py', cwd=CODE_DIR)

# Method subspace overlap: dynamic, plots all pairs found in comparison_results.json
run('python analysis/plot_method_overlap.py', cwd=CODE_DIR)

# Safety-utility overlap: DIM mean-diffs + overlaid RCO/DIM direction lines
run('python analysis/plot_safety_utility_overlap.py', cwd=CODE_DIR)

# RepInd: use RCO results if available, otherwise DIM-derived
rco_repind = CODE_DIR / 'results/geometry_repind_rco/geometry_repind_results.json'
if rco_repind.exists():
    run('python analysis/plot_geometry_repind.py --results_dir results/geometry_repind_rco', cwd=CODE_DIR)
else:
    run('python analysis/plot_geometry_repind.py', cwd=CODE_DIR)

# Prompt attack type probe figures (if probe results exist)
probe_results = CODE_DIR / 'results/probe_attack_types/results.json'
if probe_results.exists():
    run('python analysis/plot_attack_types.py', cwd=CODE_DIR)

# Sync all figures to docs/figures/ and print inventory
run('python scripts/sync_figures.py', cwd=PROJECT)
run('python scripts/inventory.py', cwd=PROJECT)

## 12. Package all results into a zip

Creates `colab_results.zip` containing all JSON results, all figures, the DIM/RCO direction tensors, the probe prompts, and the DIM direction metadata needed to re-render figures and write the report locally. Does **not** include modified-model weights (15-30 GB each); see cell 13 for that.

The zip contains:
- `results/benchmark/benchmark_results.json` - ASR, harmless compliance, and perplexity for all methods
- `results/safety_utility_overlap/safety_utility_overlap_results.json` - per-layer MSO for each method's direction vs utility subspace
- `results/method_overlap/comparison_results.json` - pairwise MSO and direction cosine similarities between methods
- `results/geometry_repind/geometry_repind_results.json` - RepInd cosine-profile change summaries
- `results/geometry_repind_rco/geometry_repind_results.json` - RepInd with RCO directions (if available)
- `results/probe_attack_types/results.json` - per-prompt projection + completions for the prompt-attack probe
- `results/probe_attack_types/prompts.json` - the actual HarmBench/WildJailbreak prompts used
- `figures/` - all PNGs synced to docs/figures/
- `dim_metadata/direction_metadata.json` - which layer DIM selected
- `directions/dim_direction.pt` - DIM refusal direction tensor
- `directions/rco_direction.pt` - RCO refusal direction tensor (if RCO ran)

Total size ~10 MB. Direct browser download via `files.download()` works fine.

In [ ]:
import shutil, zipfile
from pathlib import Path

RESULTS = PROJECT / 'code' / 'results'
DOCS_FIGS = PROJECT / 'docs' / 'figures'
DIM_RUN = PROJECT / 'code' / 'methods' / 'dim' / 'pipeline' / 'runs' / MODEL_ALIAS
DIM_DIRECTION = DIM_RUN / 'direction.pt'
RCO_DIRECTION = RESULTS / 'geometry_repind' / 'rco_direction.pt'
ZIP_PATH = PROJECT / 'colab_results.zip'

# Also discover the latest cones-repind RCO artifact (the raw 2-D cone basis)
RCO_ROOT = PROJECT / 'code' / 'methods' / 'cones-repind' / 'results' / 'rdo'
rco_candidates = list(RCO_ROOT.glob('*/local_vectors/*/*/lowest_loss_vector.pt'))
RCO_BASIS = max(rco_candidates, key=lambda p: p.stat().st_mtime) if rco_candidates else None

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    # All result JSONs (benchmark, overlap, repind, probe, etc.)
    for json_file in sorted(RESULTS.rglob('*.json')):
        zf.write(json_file, f'results/{json_file.relative_to(RESULTS)}')
        print(f'  + results/{json_file.relative_to(RESULTS)}')

    # All synced figures
    if DOCS_FIGS.exists():
        for fig in sorted(DOCS_FIGS.glob('*.png')):
            zf.write(fig, f'figures/{fig.name}')
            print(f'  + figures/{fig.name}')

    # Per-experiment figures (sync_figures.py may not have them all)
    for fig in sorted(RESULTS.rglob('*.png')):
        zf.write(fig, f'results/{fig.relative_to(RESULTS)}')

    # DIM artifacts
    dim_meta = DIM_RUN / 'direction_metadata.json'
    if dim_meta.exists():
        zf.write(dim_meta, 'dim_metadata/direction_metadata.json')
        print(f'  + dim_metadata/direction_metadata.json')
    if DIM_DIRECTION.exists():
        zf.write(DIM_DIRECTION, 'directions/dim_direction.pt')
        print(f'  + directions/dim_direction.pt')

    # RCO direction (the standalone .pt extracted by run_rco_eval.sh)
    if RCO_DIRECTION.exists():
        zf.write(RCO_DIRECTION, 'directions/rco_direction.pt')
        print(f'  + directions/rco_direction.pt')

    # RCO raw 2-D cone basis (lowest_loss_vector.pt)
    if RCO_BASIS is not None:
        zf.write(RCO_BASIS, 'directions/rco_lowest_loss_vector.pt')
        print(f'  + directions/rco_lowest_loss_vector.pt  ({RCO_BASIS.relative_to(PROJECT)})')

print(f'\nZip created: {ZIP_PATH}')
print(f'Size: {ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB')

# Download in Colab
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print('Not in Colab, skipping download. Zip is at:', ZIP_PATH)

## 13. (Optional) Save modified models to Google Drive

The DIM, ActSVD, and RCO modified models are 15-30 GB each. Direct browser download via `files.download()` is unreliable above 2 GB; instead we copy them to your Google Drive, from which you can download them with `gdrive` or `rclone` locally, or sync them via the desktop Google Drive app.

**Set `SAVE_MODELS_TO_DRIVE = True`** to enable. The cell will:
1. Mount your Google Drive (you'll see an OAuth prompt).
2. Copy each available modified-model directory to `MyDrive/nlp-project-models/`.
3. Print sizes so you can plan storage.

Free Google Drive gives 15 GB total. ActSVD alone is ~15 GB. If you need all three, you'll need a paid tier (or upload separately to Hugging Face Hub via `huggingface-cli upload`).

In [ ]:
SAVE_MODELS_TO_DRIVE = False  # flip to True to copy modified models to Google Drive

if SAVE_MODELS_TO_DRIVE:
    import shutil
    from pathlib import Path

    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
    except Exception as exc:
        raise RuntimeError(f'Google Drive mount failed: {exc}')

    DRIVE_ROOT = Path('/content/drive/MyDrive/nlp-project-models') / MODEL_ALIAS
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    # Map of source dir -> destination subdir (only includes ones that exist)
    candidates = {
        'dim_modified':    PROJECT / 'code/methods/dim/pipeline/runs' / MODEL_ALIAS / 'modified_model',
        'actsvd_modified': PROJECT / 'code/methods/actsvd/out',
        'rco_modified':    PROJECT / 'code/methods/cones-repind/results/modified_model' / f'{MODEL_ALIAS}-RCO',
    }

    def _dir_size(p: Path) -> int:
        return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

    for name, src in candidates.items():
        if not src.exists():
            print(f'[skip] {name}: not found at {src}')
            continue
        dst = DRIVE_ROOT / name
        size_gb = _dir_size(src) / 1024**3
        print(f'[copy] {name}: {size_gb:.1f} GB -> {dst}')
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'       done.')

    # Also stash the small artifacts (directions + metadata) alongside, so the
    # Drive folder is self-contained for offline analysis.
    aux = DRIVE_ROOT / 'aux'
    aux.mkdir(exist_ok=True)
    for src_path, arc_name in [
        (PROJECT / 'code/methods/dim/pipeline/runs' / MODEL_ALIAS / 'direction.pt',          'dim_direction.pt'),
        (PROJECT / 'code/methods/dim/pipeline/runs' / MODEL_ALIAS / 'direction_metadata.json','dim_direction_metadata.json'),
        (PROJECT / 'code/results/geometry_repind/rco_direction.pt',                          'rco_direction.pt'),
    ]:
        if src_path.exists():
            shutil.copy2(src_path, aux / arc_name)
            print(f'  + aux/{arc_name}')

    print(f'\nAll modified-model artifacts under: {DRIVE_ROOT}')
    print('Sync to your laptop via the Google Drive desktop app, gdrive CLI, or rclone.')
else:
    print('SAVE_MODELS_TO_DRIVE is False — skipping. Flip to True if you want the modified models locally.')

---
## Appendix A: Single-direction ablation / faster smoke test (RDO)

If you want a quicker run — to verify the pipeline end-to-end or to compare against the full cone — train a single optimized direction instead of the 2D cone. This is equivalent to the RDO configuration from Wollschläger et al. and takes ~30 min instead of ~90 min.

The tradeoff: a single direction is a weaker intervention. An adversary can craft prompts that activate refusal orthogonally to it, whereas the 2D cone removes an entire plane. For our geometric analysis the single direction is still useful as an ablation: comparing RDO (1D) vs RCO (2D) MSO scores shows whether the second cone dimension provides additional coverage of the safety or utility subspace.

After this cell completes, re-run cells 6, 7, 8, and 10 to add `RDO` to all analysis outputs alongside `RCO-Cone-2`.

In [ ]:
run('./scripts/run_rco_eval.sh', env_extra={
    'MODEL': MODEL_PATH,
    'MODE': 'direction',
    'METHOD_NAME': 'RDO',
    'WANDB_MODE': 'offline',
    'SKIP_EVAL': '0',
    'SKIP_REPIND': '0',
})

## Appendix B: Ad hoc subspace comparison

Use `scripts/compare_overlap.py` to compute MSO between any two saved `.pt` tensors without re-running the full pipeline.

In [ ]:
# DIM direction vs all candidate mean-diffs
run(
    f'python scripts/compare_overlap.py'
    f' --left code/methods/dim/pipeline/runs/{MODEL_ALIAS}/direction.pt'
    f' --right code/methods/dim/pipeline/runs/{MODEL_ALIAS}/generate_directions/mean_diffs.pt'
    f' --right-rank 8'
)

# DIM direction vs RCO direction
run(
    f'python scripts/compare_overlap.py'
    f' --left code/methods/dim/pipeline/runs/{MODEL_ALIAS}/direction.pt'
    f' --right code/results/geometry_repind/rco_direction.pt'
)